# Notebook 09: AlphaZero Curriculum Experiments

Este notebook abre uma linha experimental separada para `AlphaZero` com curriculum de oponentes, sem alterar o notebook base de `self-play` puro.


## Objetivo

O objetivo deste notebook e:

- comparar varias agendas de oponentes para `AlphaZero`;
- medir o impacto em `vs_random`, `vs_heuristic`, `vs_previous` e precisao tatica;
- manter a extensao como trilho paralelo ao notebook `05_alphazero_simplified.ipynb`.


In [ ]:
from __future__ import annotations

import json
import statistics
import sys
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "config.yaml").exists():
    raise FileNotFoundError("Could not find project root containing config.yaml")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUTPUTS = ROOT / "outputs"
CURRICULUM_OUTPUTS = OUTPUTS / "alphazero_curriculum_experiments"
CURRICULUM_OUTPUTS.mkdir(parents=True, exist_ok=True)

from connect4_rl.config import load_config
from connect4_rl.agents.learning import AlphaZeroConfig
from connect4_rl.experiments.alphazero_curriculum import (
    build_default_alphazero_curricula,
    expand_curriculum_schedule,
    train_alphazero_with_curriculum,
)

CONFIG = load_config(ROOT / "config.yaml")
NOTEBOOK_DEVICE = CONFIG.resolve_device()
if NOTEBOOK_DEVICE == "cuda":
    torch.set_default_device(NOTEBOOK_DEVICE)
print(
    {
        "torch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "device": NOTEBOOK_DEVICE,
        "outputs": str(CURRICULUM_OUTPUTS),
    }
)


## Passo 1: Configuracao da experiencia

Controlamos aqui se o notebook vai mesmo correr treinos ou apenas analisar resultados ja guardados.


In [ ]:
run_training = False

curricula = build_default_alphazero_curricula()
selected_agendas = [
    "curriculum_basic",
    "curriculum_mid_self",
    "curriculum_short_heuristic_late",
    "curriculum_probabilistic_bridge",
]
seeds = [7, 17, 27]

episodes = 180
mcts_simulations = 80
eval_mcts_simulations = 120
eval_interval = 30
eval_games = 12

experiment_config = AlphaZeroConfig(
    **{
        **asdict(CONFIG.alphazero),
        "episodes": episodes,
        "mcts_simulations": mcts_simulations,
        "eval_mcts_simulations": eval_mcts_simulations,
        "eval_interval": eval_interval,
        "eval_games": eval_games,
        "n_res_blocks": 4,
        "updates_per_episode": 1,
        "replay_warmup_games": 12,
        "tactical_eval_examples": 64,
        "device": NOTEBOOK_DEVICE,
    }
)

{
    "agendas": selected_agendas,
    "seeds": seeds,
    "episodes": experiment_config.episodes,
    "device": experiment_config.device,
    "mcts_simulations": experiment_config.mcts_simulations,
    "eval_interval": experiment_config.eval_interval,
}


## Passo 2: Inspecao das agendas

Antes de correr qualquer treino, convem verificar como cada agenda se expande em episodios reais.


In [ ]:
for agenda_name in selected_agendas:
    definition = curricula[agenda_name]
    schedule, phase_summary = expand_curriculum_schedule(experiment_config.episodes, definition, seed=seeds[0])
    print(f"=== {agenda_name} ===")
    print(definition.description)
    for item in phase_summary:
        print(item)
    print("counts:", {kind: schedule.count(kind) for kind in sorted(set(schedule))})


## Passo 3: Execucao opcional dos treinos

Quando `run_training = True`, esta celula corre todas as agendas e seeds selecionadas.


In [ ]:
results = []

if run_training:
    for agenda_name in selected_agendas:
        for seed in seeds:
            run_config = AlphaZeroConfig(**{**asdict(experiment_config), "seed": seed})
            checkpoint_dir = CURRICULUM_OUTPUTS / f"{agenda_name}_seed_{seed}"
            print(f"=== Running {agenda_name} | seed={seed} ===")
            _agent, metrics = train_alphazero_with_curriculum(
                curricula[agenda_name],
                run_config,
                checkpoint_dir=checkpoint_dir,
            )
            last_eval = metrics.evaluation[-1] if metrics.evaluation else {}
            last_tactical = metrics.tactical_accuracy[-1] if metrics.tactical_accuracy else {}
            print("Last eval:", last_eval)
            print("Last tactical accuracy:", last_tactical)
            results.append(
                {
                    "agenda": agenda_name,
                    "seed": seed,
                    "best_score": metrics.best_score,
                    "last_eval": last_eval,
                    "last_tactical": last_tactical,
                    "best_checkpoint_path": metrics.best_checkpoint_path,
                }
            )
else:
    print("Training skipped. Set run_training = True to launch experiments.")

results[:2]


## Passo 4: Carregamento dos resultados guardados

A analise abaixo funciona mesmo sem treino novo, desde que existam corridas guardadas em `outputs/alphazero_curriculum_experiments/`.


In [ ]:
def load_curriculum_runs(root: Path) -> list[dict]:
    runs = []
    for metrics_path in sorted(root.glob("*_seed_*/metrics_final.json")):
        data = json.loads(metrics_path.read_text(encoding="utf-8"))
        run_name = metrics_path.parent.name
        family = run_name.split("_seed_")[0] if "_seed_" in run_name else run_name
        runs.append(
            {
                "run_name": run_name,
                "family": family,
                "path": metrics_path,
                "data": data,
            }
        )
    return runs

all_runs = load_curriculum_runs(CURRICULUM_OUTPUTS)
[(run["run_name"], len(run["data"].get("evaluation", []))) for run in all_runs]


## Passo 5: Agregacao por agenda

Resumimos cada familia de agenda usando a ultima avaliacao de cada seed.


In [ ]:
def summarize_runs(runs: list[dict]) -> list[dict]:
    grouped: dict[str, list[dict]] = {}
    for run in runs:
        grouped.setdefault(run["family"], []).append(run)

    rows = []
    for family, family_runs in sorted(grouped.items()):
        best_scores = [float(run["data"].get("best_score", float("-inf"))) for run in family_runs]
        final_random = []
        final_heuristic = []
        final_previous = []
        final_tactical = []
        for run in family_runs:
            evaluation = run["data"].get("evaluation", [])
            if evaluation:
                final_random.append(float(evaluation[-1].get("vs_random_win_rate", 0.0)))
                final_heuristic.append(float(evaluation[-1].get("vs_heuristic_win_rate", 0.0)))
                final_previous.append(float(evaluation[-1].get("vs_previous_win_rate", 0.0)))
            tactical = run["data"].get("tactical_accuracy", [])
            if tactical:
                final_tactical.append(float(tactical[-1].get("accuracy", 0.0)))

        rows.append(
            {
                "agenda": family,
                "num_runs": len(family_runs),
                "mean_best_score": round(statistics.fmean(best_scores), 4) if best_scores else None,
                "mean_vs_random": round(statistics.fmean(final_random), 4) if final_random else None,
                "mean_vs_heuristic": round(statistics.fmean(final_heuristic), 4) if final_heuristic else None,
                "mean_vs_previous": round(statistics.fmean(final_previous), 4) if final_previous else None,
                "mean_tactical": round(statistics.fmean(final_tactical), 4) if final_tactical else None,
            }
        )
    return rows

summary_rows = summarize_runs(all_runs)
summary_rows


## Passo 6: Visualizacao rapida

Os graficos seguintes ajudam a perceber se alguma agenda ganhou tracao contra `heuristic` sem sacrificar demasiado o resto.


In [ ]:
if summary_rows:
    agendas = [row["agenda"] for row in summary_rows]
    vs_random = [row["mean_vs_random"] or 0.0 for row in summary_rows]
    vs_heuristic = [row["mean_vs_heuristic"] or 0.0 for row in summary_rows]
    tactical = [row["mean_tactical"] or 0.0 for row in summary_rows]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].bar(agendas, vs_random)
    axes[0].set_title("Final win rate vs random")
    axes[0].tick_params(axis="x", rotation=25)

    axes[1].bar(agendas, vs_heuristic)
    axes[1].set_title("Final win rate vs heuristic")
    axes[1].tick_params(axis="x", rotation=25)

    axes[2].bar(agendas, tactical)
    axes[2].set_title("Final tactical accuracy")
    axes[2].tick_params(axis="x", rotation=25)

    plt.tight_layout()
    plt.show()
else:
    print("No runs found yet.")


## Leituras esperadas

- se uma agenda melhora cedo contra `random` mas continua a zero contra `heuristic`, o curriculum provavelmente nao esta a transferir para jogo mais estruturado;
- se a precisao tatica sobe mas o `vs_previous` cai, pode haver aprendizagem local mas pouca estabilidade global;
- se a fase mista funcionar melhor do que blocos rigidos, isso sugere que a transicao gradual e mais natural para este `AlphaZero` simplificado.
